# BeeNoise — local pipeline check

Runs every backend stage — denoise, diarize, speaker ID, transcribe, merge —
individually and then as the full pipeline, against your own machine's
`.venv`. Nothing here uploads anywhere or starts a public server; this is
purely for verifying (and showing) that the backend actually works.

**Prerequisites** (see [QUICKSTART.md](QUICKSTART.md) if you haven't done these):
- The project's `.venv` exists and has every dependency installed.
- This notebook's kernel is that `.venv`'s Python — checked in the next cell.
  If you haven't registered it as a Jupyter kernel yet, run once from an
  activated `.venv`:
  ```
  pip install -r requirements-notebook.txt
  python -m ipykernel install --user --name beenoise --display-name "BeeNoise (.venv)"
  ```
  then pick **BeeNoise (.venv)** as this notebook's kernel.
- `.env` has a working `HF_TOKEN` if you want to exercise the `pyannote`
  diarizer (default). Without one, set `DIARIZER = "ecapa_cluster"` below
  instead — no token needed.
- At least one enrolled speaker helps (Stage 3 has more to show), but the
  "Enroll a speaker" section below can create one for you if you don't.

## 0. Confirm the kernel is actually this project's `.venv`

In [ ]:
import pathlib
import sys

venv_python = pathlib.Path.cwd() / '.venv' / 'Scripts' / 'python.exe'
if pathlib.Path(sys.executable).resolve() != venv_python.resolve():
    print(f'WARNING: this kernel is\n  {sys.executable}\nnot the project venv\n  {venv_python}')
    print('Fix: Kernel menu -> Change kernel -> "BeeNoise (.venv)" (see the setup cell above).')
else:
    print('OK — running inside the project .venv.')

## 1. Environment check

Reuses the same checks `python -m src.check_env` runs.

In [ ]:
from src.check_env import check_imports

check_imports()

## 2. Pick which diarizer to use

`pyannote` (default) needs `HF_TOKEN` set up and the gated model terms
accepted — see QUICKSTART.md step 3. `ecapa_cluster` is this project's own
baseline diarizer and needs nothing extra — use it if you haven't set up a
token, or just want a faster/offline run.

In [ ]:
DIARIZER = "pyannote"  # or "ecapa_cluster"

## 3. Get a test clip

Run **one** of the next two cells, not both.

**Option A — record ~5-10s from your microphone** (say a full sentence or two):

In [ ]:
from src.audio_io import record_until_enter, save_wav
from src.config import CFG

sr = CFG['audio']['output_sr']
input('Press Enter, then speak — press Enter again to stop recording...')
clip = record_until_enter(sr, max_seconds=15)
save_wav('notebook_test_clip.wav', clip, sr)
CLIP_PATH = 'notebook_test_clip.wav'
print(f'Recorded {len(clip)/sr:.1f}s -> {CLIP_PATH}')

**Option B — use an existing file instead** (edit the path, then run this cell):

In [ ]:
import pathlib

CLIP_PATH = r"C:\path\to\your\clip.mp4"  # <-- edit this

assert pathlib.Path(CLIP_PATH).exists(), f'No file at {CLIP_PATH} — check the path.'
print(f'Using {CLIP_PATH}')

## 4. Enroll a speaker

Skip this section if you've already enrolled someone (`python -m src.enroll`)
and just want to check the pipeline recognises them. Otherwise, run **one**
of the next two cells to enroll whoever is in your test clip — that way
Stage 3 below has an actual name to find, not just anonymous speakers.

In [ ]:
# Option A — enroll from the microphone (~30-60s of reading a paragraph aloud)
from src.audio_io import record_until_enter
from src.config import CFG
from src.enroll import PARAGRAPHS, enroll_audio

sr = CFG['audio']['analysis_sr']
NAME = "YourName"  # <-- edit this
print(PARAGRAPHS['en'])
input('\nPress Enter, then read the paragraph above — press Enter again to stop...')
enroll_clip = record_until_enter(sr, max_seconds=90)
profile = enroll_audio(NAME, enroll_clip, sr)
print(f'Enrolled {profile.name}: {profile.speech_sec:.1f}s speech, spread={profile.spread:.3f}')

In [ ]:
# Option B — enroll from an existing recording
from src.audio_io import load_audio
from src.config import CFG
from src.enroll import enroll_audio

sr = CFG['audio']['analysis_sr']
NAME = "YourName"  # <-- edit this
ENROLL_FILE = r"C:\path\to\a_paragraph_reading.wav"  # <-- edit this
profile = enroll_audio(NAME, load_audio(ENROLL_FILE, sr), sr)
print(f'Enrolled {profile.name}: {profile.speech_sec:.1f}s speech, spread={profile.spread:.3f}')

In [ ]:
from src import profiles

for p in profiles.get_all():
    print(f'{p.name:<20} speech={p.speech_sec:5.1f}s  chunks={p.n_chunks:3d}  spread={p.spread:.3f}')

## 5. Run each stage by hand

This is the actual point of this notebook — calling each backend module
directly, the same functions `src/main.py` calls internally, so you can see
every stage's raw output on its own.

### Stage 1 — Denoise (`src/denoise.py`)

In [ ]:
from IPython.display import Audio, display

from src.audio_io import load_audio, resample
from src.config import CFG
from src.denoise import denoise

o_sr, a_sr = CFG['audio']['output_sr'], CFG['audio']['analysis_sr']
raw = load_audio(CLIP_PATH, o_sr)
clean, clean_sr = denoise(raw, o_sr, CFG['denoise']['backend'])
clean16 = resample(clean, clean_sr, a_sr)  # -> analysis rate for the stages below

print(f"Backend: {CFG['denoise']['backend']}  |  {len(raw)/o_sr:.1f}s")
print('Before:')
display(Audio(raw, rate=o_sr))
print('After:')
display(Audio(clean, rate=clean_sr))

### Stage 2 — Diarize (`src/diarize.py`)

In [ ]:
from src.diarize import diarize

segments = diarize(clean16, a_sr, DIARIZER)
print(f'{len(segments)} turn(s), {len({s.speaker for s in segments})} speaker(s):')
for s in segments:
    print(f'  {s.start:6.2f}s - {s.end:6.2f}s  {s.speaker}')

### Stage 3 — Speaker ID (`src/speaker_id.py`)

In [ ]:
from src.speaker_id import identify_clusters, relabel

matches = identify_clusters(clean16, a_sr, segments)
for label, m in matches.items():
    score = f'{m.score:.3f}' if m.score is not None else 'n/a'
    tag = 'enrolled match' if m.enrolled else 'unenrolled'
    print(f'  {label:<14} -> {m.name:<16} ({tag}, best candidate={m.best_candidate}, cos={score})')

named_segments = relabel(segments, matches)

### Stage 4 — Transcribe (`src/stt.py`)

In [ ]:
from src.stt import transcribe_words

words, language = transcribe_words(clean16, language='en')
print(f'Detected language: {language}')
print('Text:', ''.join(w.text for w in words).strip())
print()
print('First few words with timestamps:')
for w in words[:8]:
    print(f'  {w.start:5.2f}s - {w.end:5.2f}s  {w.text!r}')

### Stage 5 — Merge (`src/merge.py`)

In [ ]:
from src.merge import assign_speakers, build_cues, to_srt

labelled_words = assign_speakers(words, named_segments)
cues = build_cues(labelled_words)
print(to_srt(cues))

## 6. The full pipeline in one call

Same five stages, but through the actual entry point `python -m src.main`
uses (`run_file`), writing every output file to `outputs/<clip name>/` —
this is the same code path the CLI and the Gradio UI both call.

In [ ]:
from src.main import Options, run_file

opts = Options(diarizer=DIARIZER, num_speakers=1)  # set num_speakers=None if unsure how many people are in the clip
paths = run_file(CLIP_PATH, opts=opts)

In [ ]:
from IPython.display import Audio, Video, display

print('Denoised audio:')
display(Audio(str(paths['denoised'])))

print('Subtitles:')
print(paths['srt'].read_text(encoding='utf-8'))

if 'video' in paths:
    print('Subtitled video:')
    display(Video(str(paths['video']), embed=True, width=480))

## 7. Run the automated test suite

The unit tests for everything that doesn't need a model or a mic — pure
logic (subtitle building, RTTM parsing, speaker assignment math). Should
finish in well under a second.

In [ ]:
import subprocess
import sys

result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

---
## What's not covered here

- **The interactive web UI** — this notebook calls the backend directly;
  for the Gradio UI (Enroll / Transcribe / Speakers tabs) on your own
  machine, run in a terminal: `python -m src.demo_ui`, then open
  http://127.0.0.1:7860.
- **Evaluation metrics** (denoising quality, DER, speaker ID accuracy, WER)
  — those are the `eval/*.py` scripts, run once you have real held-out test
  data (see [data/README.md](data/README.md)), not part of this smoke test.